In [ ]:
# BERT를 사용하려면
# transformers library를 설치해야 해요!
# KOBERT를 이용할예정인데.. 이 KOBERT를 이용하려면 Tokenizer가 있어야 해요!
# kobert-transformers library안에 있는 Tokenizer를 이용할꺼예요!

In [3]:
from transformers import BertModel
import torch
from kobert_transformers import get_tokenizer

# KOBERT 사전학습 모델 이름
model_name = 'skt/kobert-base-v1'

tokenizer = get_tokenizer()
bert_model = BertModel.from_pretrained(model_name)

In [9]:
# 함수를 하나 만들꺼예요. tokenizing하는 함수를 만들어요!
def tokenize_text(texts, tokenizer, max_len=64):
    return tokenizer(
        texts,   # 입력문장, 문장 리스트
        padding='max_length',  # 시퀀스의 기리이를 max_length에 맞춰 패딩
        truncation=True, # 문장이 max_length보다 길면 잘라냄.
        max_length=max_len,
        return_tensors='pt',
        return_token_type_ids=True
    )
texts = ['나는 학교에 간다', '오늘 날씨가 너무 좋아요']

tokenized = tokenize_text(texts,
                          tokenizer,
                          max_len=64)
# BERT모델에서 입력데이터를 tokenizing하면
# 결과를 3개 얻을 수 있어요!
# tokenized.input_ids
print(tokenized.input_ids.shape)  # 문장 2개를 각각 숫자로 변환. 단어사전을 기준으로
                                  # 각 토큰을 숫자로 변경(정수). 그리고 길이를 64개로 패딩
                                  # shape : (2,64)
print(tokenized.input_ids)
print(tokenized.token_type_ids)   # 우리 예제에서는 필요가 없어요!
                                  # 문장이 2개 입력으로 들어가는 경우 그걸 구분해주기 위해서
                                  # 존재하는 값이예요!
print(tokenized.attention_mask)   # 실제 토큰인지 패딩인지를 알려주는 역할


torch.Size([2, 64])
tensor([[   2, 1375, 4949, 6896,  517, 5338,    3,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1],
        [   2, 3419, 1408, 5330, 1458, 4207, 6999,    3,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0,

In [28]:
# 네이버 영화 댓글 감정분류 예제를 기존에는 RNN, LSTM을 이용해서
# 구현했는데 이번에는 BERT를 이용해서 구현해 볼꺼예요!
# RNN, LSTM 할때는 Mecab이라는 형태소 분석기를 우리가 직접 사용.
# Tokenizer를 이용해서 단어사전도 직접 만들었어요!
# 불필요한 특수문자 제거, 불용어 삭제처리도 진행
# 결국 Embedding까지 우리가 직접 처리해서 그 결과를 모델에 입력해서 학습을 진행했어요!

%reset -f

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import BertModel
from kobert_transformers import get_tokenizer
from sklearn.model_selection import train_test_split
import datetime

In [29]:
# pandas의 DataFrame에 대해 반복작업을 수행할 때 progressbar로 
# 진행상황을 보여주기 위해
tqdm.pandas()

# GPU 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [30]:
# 1. 데이터 로딩 및 전처리
import urllib.request

if not os.path.exists('ratings_train.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt',
        'ratings_train.txt'
    )

train_df = pd.read_csv('ratings_train.txt', sep='\t')
# display(train_df.head())

# RNN, LSTM과는 다르게 특수문자를 제거하지 않을꺼예요!
# 대신 결측치는 5개 있었는데 이건 처리해야해요!
# 불용어처리 하지 않아요!
# 가능한 원어 그대로 입력데이터로 사용할꺼예요!
train_df = train_df.dropna()  # 결측치 제거
train_df['document'] = train_df['document'].str.strip()
display(train_df.shape) # (149995, 3)

# 2. 토크나이저와 모델을 생성
model_name = 'skt/kobert-base-v1'

tokenizer = get_tokenizer()
bert_model = BertModel.from_pretrained(model_name)

(149995, 3)

/home/moon9342/anaconda3/envs/nlp_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [31]:
# 3. 데이터셋 준비
# pytorch의 Dataset기능을 이어받아 확장된 Dataset기능을 만들꺼예요!
class NSMCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        # 변수(속성)을 이용해서 전달된 값을 저장
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # 인덱스를 이용해서 댓글과 해당 댓글에 대한 label을 추출하는 기능을 정의
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        # dictionary로 처리된 결과를 return할꺼예요!
        # 결론적으로 말하면
        # 우리 데이터셋으로부터 데이터를 꺼내면 아래와 같은 형태로
        # 데이터를 변환시켜서 전달하려는 거예요!
        return {
           'input_ids': encoding['input_ids'].flatten(),
           'attention_mask' : encoding['attention_mask'].flatten(),
           'label' : torch.tensor(label, dtype=torch.float)
        }

In [37]:
# 4. 모델 정의
class KoBERTClassifier(nn.Module):
    def __init__(self, bert_model):
        super().__init__()  # 상위 클래스 초기화
        self.bert = bert_model
        self.hidden_size = self.bert.config.hidden_size 

        self.classifier = nn.Linear(self.hidden_size, 1)  
        
    def forward(self, input_ids, attention_mask):  # BERT모델의 입력
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # [CLS] Token의 결과값 사용
        # outputs => (batch_size, seq_len, hidden_size)
        cls_output = outputs.last_hidden_state[:,0,:]  # CLS token에 대한 결과값.
        # 문장들을 KOBERT를 이용해서 숫자로 바꾼 최종 결과값들.
        logits = self.classifier(cls_output)

        output = torch.sigmoid(logits)

        return output.squeeze()

In [38]:
# 5. 학습함수
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()

    for batch in tqdm(data_loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, label)

        loss.backward()  # 역전파
        optimizer.step()

In [39]:
# 6. 검증함수
def eval_epoch(model, data_loader, criterion, device):
    model.eval()

    with torch_no_grad():
        for batch in tqdm(data_loader, desc='Validation'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label = batch['label'].to(device)
    
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, label)

In [41]:
# 7. 데이터 준비
max_len = 64
batch_size = 128

# 학습데이터와 검증데이터를 분리
x_data_train, x_data_val, y_data_train, y_data_val = \
train_test_split(
    train_df['document'].values,
    train_df['label'].values,
    test_size=0.2,
    stratify=train_df['label'].values
)

# 데이터셋 생성(전체데이터에 대한 데이터셋)
# 여기에서 데이터를 추출하면 한건 한건씩 나와요!
# 이걸 그대로 사용하면 한번에 하나의 문장만 모델에 입력이 되요!
# 속도가 느려져... 배치처리를 해야 해요!
train_dataset = NSMCDataset(x_data_train,
                            y_data_train,
                            tokenizer,
                            max_len)
val_dataset = NSMCDataset(x_data_val,
                          y_data_val,
                          tokenizer,
                          max_len)

# 데이터로더를 생성(배치처리를 하기 위해)
train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(val_dataset,
                        batch_size=batch_size)

In [42]:
model = KoBERTClassifier(bert_model)  # 동작하는 모델을 생성
model.to(device)

optimizer = optim.Adam(model.parameters(),
                       lr=1e-3)
criterion = nn.BCELoss()

In [43]:
# 9. 학습
epochs = 20

print('학습시작')
for epoch in range(epochs):
    print(f'{epoch+1} / {epochs}')

    train_epoch(model, train_loader, optimizer, criterion, device)

    eval_epoch(model, val_loader, optimizer, criterion, device)

학습시작
1 / 20


Training: 100%|███████████████████████████████████████████████████████████████| 938/938 [15:21<00:00,  1.02it/s]


NameError: name 'val_epoch' is not defined

In [ ]:
# 학습이 종료되면
# 예측작업을 해 보면 되요!
def predict_sentiment(text, model, tokenizer, device, max_len=64):
    model.eval()

    encoding = tokenizer(text,
                        truncation=True,
                        padding='max_length',
                        max_length=max_len,
                        return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probability = output.item()

    sentiment = "긍정" if probability >= 0.5 else "부정"
    return sentiment, probability

test_texts = ['이 영화 정말 재미있어요!',
              '완전 시간 낭비 였어요!',
              '그저 그래요. 특별한건 없어요!']

for text in test_texts:
    sentiment, prob = predict_sentiment(text,
                                        model,
                                        tokenizer,
                                        device)
    print(f'텍스트 : {text}')
    print(f'감정 : {sentiment}, 확률 : {prob}')